# Pathway interference: a CORE network from receptor to gene expression, and how a TARGET's inhibition changes it

**Inputs: `CORE` (the gene whose pathway you care about — usually a receptor or a disease-causal gene) and `TARGET` (the gene a drug would inhibit).**

1. **CORE network** — downstream walk from CORE on the full signed graph (SIGNOR + OmniPath, commercial-licence resources): intermediates → transcription factors → DoRothEA target genes. Drawn as a **cell cross-section**: ligands / membrane at the top, cytosolic intermediates in the middle (row = hop), nucleus with the TFs at the bottom.
2. **TARGET network** — the same walk from TARGET (same graph, same thresholds), drawn the same way (`cell_flow_target.html`).
3. **Overlap** — the TARGET network laid over the CORE cross-section: shared nodes (purple), CORE nodes receiving an edge from the TARGET network (tan), and the TARGET-network nodes that act on the CORE network in the left margin. This is *where* the pathways meet (`overlap_nodes.csv`, `overlap_edges.csv`, Jaccard).
4. **Interference by TARGET inhibition** — a "TARGET inhibited" perturbation enters the CORE network through every edge from TARGET's side (TARGET itself, or a node TARGET reaches) into a CORE-network node; entry sign = −sign(TARGET→u) × effect(u→v). It then propagates along CORE's own signed edges. Every CORE node gets a **delta**: < 0 = predicted to decrease when TARGET is inhibited, > 0 = to increase (TARGET was a brake); DoRothEA genes inherit delta(TF) × mode of regulation.

Outputs in `runs/<time>_pathway_<CORE>_vs_<TARGET>/`: `downstream/` (CORE tables and flow views), `interference/cell_flow.html`, `interference/cell_flow_inhibition.html`, `flow_change.csv`, `flow_change_genes.csv`, `entry_edges.csv`, `summary.json`.

## 1. Parameters

In [ ]:
CORE = "FGFR3"         # gene whose pathway is analysed (receptor / disease-causal gene)
TARGET = "MAPK1"       # gene to be inhibited
DEMO = False           # False = your databases from target_net.yaml; True = offline curated demo graph
CONFIG = None          # None = find target_net.yaml in this folder or any parent, else ~/.target_net.yaml; or give a path
TISSUE = None          # HPA tissue name to restrict the graph to expressed genes (e.g. "endothelial cells" is not a tissue; use e.g. "lung")
MAX_DEPTH = MIN_SCORE = HUB_PENALTY = None   # None = values from the YAML walk: section
TARGET_NET_DIR = ".."
FOCUS = None           # genes to highlight in the baseline network view (plain text, list, file or dict gene -> value)

## 2. Setup

In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys
sys.path.insert(0, os.path.abspath(TARGET_NET_DIR))
from targetnet.api import research_pathway_interference, summarize_pathway_interference, figure, legend
from targetnet.plots import use_cdn_renderer
use_cdn_renderer()
import pandas as pd
pd.set_option("display.max_colwidth", 80); pd.set_option("display.width", 180)
OUT_ROOT = os.path.join(TARGET_NET_DIR, "runs")

## 3. Run

In [ ]:
p = research_pathway_interference(CORE, TARGET, config=CONFIG, demo=DEMO, tissue=TISSUE, max_depth=MAX_DEPTH, min_score=MIN_SCORE,
                                  hub_penalty=HUB_PENALTY, out=OUT_ROOT)
print(summarize_pathway_interference(p))
print("outputs:", p["run_dir"])

## 4. The CORE network as a cell cross-section
Top: membrane / ligands. Middle: cytosolic intermediates, one row per hop. Bottom: nucleus with the transcription factors and, under each, the number of DoRothEA target genes (hover for the list). Outline green = activated by CORE, red = inhibited.

In [ ]:
figure(p["views"]["cell_flow"], focus=FOCUS).show()

## 4b. The TARGET network, same calculation
Identical walk and layout for TARGET, so the two networks are comparable one to one.

In [ ]:
figure(p["views"]["cell_flow_target"]).show()

## 4c. Overlap: where the TARGET network meets the CORE network
Purple = reached by both walks (shared), tan = CORE node that receives an edge from the TARGET network, grey = CORE only; orange margin = TARGET and its network nodes outside the CORE network that act on it (brown edges, hover for mechanism / residue).

In [ ]:
figure(p["views"]["cell_flow_overlap"]).show()

In [ ]:
# shared nodes with hops / signs from both seeds, and the edges from the TARGET network into the CORE network
display(p["overlap_nodes"]); p["overlap_edges"].head(30)

## 5. Direction: How the flow changes when TARGET is inhibited
Same layout. **Blue = decreases, red = increases, grey = unchanged** when TARGET is inhibited; size = |delta|; black outline = direct entry point. Orange nodes in the left margin are TARGET (diamond) and the TARGET-side nodes acting on the network; their straight edges carry the sign of the effect on the receiving node (hover the midpoint for mechanism and residue).

Note: entry strengths use TARGET's influence computed with CORE removed, so a node TARGET reaches only *through* CORE is not counted twice (once as shared, once by propagation from CORE).

In [ ]:
figure(p["views"]["cell_flow_inhibition"]).show()

## 6. Tables
`flow_change`: every CORE node with delta, direction, entry (direct / propagated) and via (TARGET-side actors). `entry_edges`: the mechanistic entry points with residues — the rows to validate by phosphoproteomics (CORE stimulation ± TARGET inhibitor). `flow_change_genes`: DoRothEA genes predicted down / up.

In [ ]:
fc = p["flow_change"]
fc[fc.direction != "unchanged"][["node","hops_from_core","is_tf","delta","direction","entry","via"]].head(30)

In [ ]:
p["entry_edges"][["from_node","to_node","hops_target_to_from","sign_target_to_from","edge_effect","mechanism","residue","effect_of_target_inhibition_on_to","entry_strength"]].head(30) if len(p["entry_edges"]) else print(f"no entry edge: {TARGET}'s reach does not touch the {CORE} network at these walk settings")

In [ ]:
p["flow_change_genes"].head(30) if len(p["flow_change_genes"]) else print("no DoRothEA gene changes (no TF of the CORE network is affected)")

## 7. How to interpret
1. **Entry edges first.** They are the mechanistic claim: "TARGET → … → u acts on v (residue R)". A direct entry at hop 1–2 of the CORE network with a residue is testable; a propagated change five hops away is an inference.
2. **Direction at the TFs is the pharmacological statement.** TFs that go *down* on TARGET inhibition are the part of CORE's output the drug would silence; TFs that go *up* are feedback releases (ERK inhibition relieving SOS1/GRB2/FRS2 brakes is the classic one) — the source of rebound and the argument for combinations.
3. **flow-change index** = fraction of CORE's TFs that change. Compare across candidate TARGETs for the same CORE (the cross-talk screen does the ranking across a list; this notebook is the detailed view of one pair).
4. **Magnitudes are relative** (products of database weights). Use them to order nodes within a run, never to compare two runs or to claim fold changes.
5. **Context.** Edges are context-free; set `TISSUE` to restrict the graph to genes expressed in the tissue of interest, and remember a TF counted as 'decreased' may not be expressed in your cell model at all.